# A1 — Catalog Enrichment / doc2query (Gemini)

One-time, resumable, cached enrichment of the 47k-track catalog: for each track, Gemini writes short retrieval queries/blurb that get appended to its doc (closes the conversational↔metadata vocabulary gap — the top recall lever per P0). Output: an enriched parquet on Drive consumed by `Catalog.id_to_metadata(enriched=True)`. Spec: `30_A1_catalog_assets.md`.

## 1. Drive + Gemini key (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'; os.environ['HF_HOME']=f'{DRIVE}/hf_cache'
OUT=f'{DRIVE}/outputs'; os.makedirs(OUT,exist_ok=True); os.makedirs(os.environ['HF_HOME'],exist_ok=True)
GEMINI_KEY = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
if not GEMINI_KEY:
    try: GEMINI_KEY = userdata.get('GEMINI_API_KEY')   # fallback to Colab Secrets
    except Exception: GEMINI_KEY = None
assert GEMINI_KEY, 'Set GEMINI_API_KEY (env var) or a Colab secret before running'
print('Gemini key loaded:', bool(GEMINI_KEY), '| length', len(GEMINI_KEY))
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=t
    from huggingface_hub import login; login(t)
except Exception as e: print('no HF_TOKEN:', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s pandas numpy google-genai
import sys; sys.path.insert(0,'.')

## 3. Config

In [ ]:
MODEL='gemini-2.5-flash-lite'   # 2.0-flash-lite shut down 2026-06-01
N_REQUESTS=4
LIMIT=0          # *** SMOKE TEST knob *** set to e.g. 200 to enrich only 200 tracks (~1 min, cents); 0 = full 47k
CHECKPOINT=2000
CONCURRENCY=64   # in-flight async requests for the full run (paid tier) — raise toward your RPM
WORKERS=16       # threads for the small gold-targeted A/B only (cell 7)
ORG='talkpl-ai'

## 4. Load catalog + few-shot style anchors (real train utterances) + Gemini generate_fn

In [ ]:
from google import genai
from google.genai import types
import random
from datasets import load_dataset
from mcrs.enrich.doc2query import build_enrich_prompt, enriched_document
from mcrs.enrich.retry import call_with_retry
from mcrs.data.ids import canonical_track_id
rows = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')

# Few-shot STYLE anchors: real listener queries from the TRAIN split (never dev/blind),
# stratified by conversation_goal.category so the synthetic queries cover the real styles.
train = load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='train')
rng = random.Random(42); by_cat = {}
for s in train.select(range(min(2000, len(train)))):
    g = (s.get('conversation_goal') or {}).get('category') or 'other'
    utts = [e['content'] for e in s['conversations'] if e['role']=='user']
    if utts: by_cat.setdefault(g, []).append(utts[0])
EXAMPLES = []
for g, utts in by_cat.items(): EXAMPLES += rng.sample(utts, min(2, len(utts)))
EXAMPLES = EXAMPLES[:10]
print(f'{len(EXAMPLES)} style exemplars across {len(by_cat)} goal categories')

# NEW google-genai SDK: opens genuine concurrent HTTPS connections (the old google-generativeai
# REST transport serialized our threads through the ::1 bridge -> ~1 req in flight, 1.2 it/s).
client = genai.Client(api_key=GEMINI_KEY)
SYSTEM, _ = build_enrich_prompt(rows[0], n_requests=N_REQUESTS, examples=EXAMPLES)
import hashlib
_tag = hashlib.sha256((MODEL + SYSTEM + str(N_REQUESTS)).encode()).hexdigest()[:8]
OUT_PARQUET = f'{OUT}/catalog_enriched_{_tag}.parquet'   # prompt/model-aware cache key
print('enriched-cache ->', OUT_PARQUET)
CFG = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=256, temperature=0.7)

def _retryable(e):
    s = str(e).lower()
    return any(x in s for x in ('503','429','500','unavailable','overloaded',
                                'deadline','timeout','rate limit','resource','connection'))
def _safe_text(resp):
    try: return (resp.text or '').strip()
    except Exception: return ''        # blocked / no candidate
def gen(meta):   # sync — used by the small gold-targeted A/B (cell 7); full run uses async below
    _, user = build_enrich_prompt(meta, n_requests=N_REQUESTS, examples=EXAMPLES)
    def _call():
        txt = _safe_text(client.models.generate_content(model=MODEL, contents=user, config=CFG))
        if not txt: raise RuntimeError('empty/blocked response')   # non-retryable -> marked failed
        return txt
    return call_with_retry(_call, max_attempts=6, base_delay=1.0, max_delay=30.0, is_retryable=_retryable)

import time as _tt
_t0=_tt.time(); _txt=gen(rows[0])
print('single-call latency:', round(_tt.time()-_t0,2), 's | sample:', _txt[:60])
# ~0.5-1s => direct (good). If the full run's it/s ~= 1/latency, concurrency isn't happening
# (local proxy/VPN serializing connections) -> run on Colab.

## 5. Resumable enrich loop — async, true concurrency (checkpoint to Drive)
Keeps up to `CONCURRENCY` requests in flight via the async client, so throughput is bounded by your paid-tier RPM rather than per-request latency. Safe to interrupt: progress is checkpointed to the parquet cache and the next run resumes from it.

In [ ]:
import os, time, asyncio, pandas as pd
from tqdm.auto import tqdm

done = {}
if os.path.exists(OUT_PARQUET):
    prev = pd.read_parquet(OUT_PARQUET); done = dict(zip(prev['track_id'], prev['enriched_doc']))
    print('resuming:', len(done), 'already enriched')
all_rows = list(rows)[:LIMIT] if LIMIT else list(rows)
pending = [r for r in all_rows if canonical_track_id(r['track_id']) not in done]
print(len(pending), 'pending of', len(all_rows))

def _flush():
    pd.DataFrame({'track_id':list(done), 'enriched_doc':list(done.values())}).to_parquet(OUT_PARQUET, index=False)

_sem = asyncio.Semaphore(CONCURRENCY)
async def _one(r):
    tid = canonical_track_id(r['track_id'])
    _, user = build_enrich_prompt(r, n_requests=N_REQUESTS, examples=EXAMPLES)
    delay = 1.0
    async with _sem:                                   # cap in-flight requests at CONCURRENCY
        for attempt in range(6):
            try:
                resp = await client.aio.models.generate_content(model=MODEL, contents=user, config=CFG)
                txt = _safe_text(resp)
                return (tid, enriched_document(r, txt)) if txt else (tid, None)
            except Exception as e:
                if attempt == 5 or not _retryable(e): return tid, None   # persistent fail -> NOT cached
                await asyncio.sleep(delay); delay = min(delay*2, 30.0)
    return tid, None

async def run():
    fails = 0; t0 = time.time()
    tasks = [asyncio.create_task(_one(r)) for r in pending]
    try:
        for i, fut in enumerate(tqdm(asyncio.as_completed(tasks), total=len(tasks)), 1):
            tid, doc = await fut
            if doc: done[tid] = doc
            else: fails += 1
            if i % CHECKPOINT == 0:
                _flush(); rate = i/max(1e-9, time.time()-t0)
                print(f'  checkpoint {i} | {rate:.1f}/s | ETA {(len(pending)-i)/max(rate,1e-9)/60:.0f} min | failed {fails}')
    finally:
        _flush()
    print('DONE ->', OUT_PARQUET, '| enriched', len(done), '| failed (re-run to retry):', fails)

await run()   # Jupyter/Colab support top-level await; in a plain .py use asyncio.run(run())

## 5b. Eyeball sample enriched docs (do the generated queries look like real listener queries?)

In [ ]:
import pandas as pd, textwrap
df = pd.read_parquet(OUT_PARQUET)
for _, r in df.head(6).iterrows():
    print('TRACK', str(r['track_id'])[:8])
    print(textwrap.fill(r['enriched_doc'], 110)); print()
# The text after ' | ' is the doc2query expansion — it should read like real conversational
# queries (mood/era/genre/similar-artist/use-case), not generic keywords.

## 6. Validate: raw vs enriched BM25 recall@100 (100 dev sessions, CPU)

In [ ]:
from mcrs.data.catalog import Catalog
from mcrs.data.conversations import Conversations
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.eval.probe import recall_ceiling
enr = dict(zip(pd.read_parquet(OUT_PARQUET)['track_id'], pd.read_parquet(OUT_PARQUET)['enriched_doc']))
cat_raw = Catalog(rows); cat_enr = Catalog(rows, enriched_docs=enr)
conv = Conversations(load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='test').select(range(100)))
turns=list(conv.turns()); qs=[QueryBuilder().build(t).text for t in turns]
golds=[conv.gold(t.session_id,t.turn_number) for t in turns]
raw = BM25Channel(cat_raw, enriched=False).batch_text_to_item_retrieval(qs, 200)
enrl= BM25Channel(cat_enr, enriched=True ).batch_text_to_item_retrieval(qs, 200)
r_raw=recall_ceiling({'bm25':raw}, golds, ks=[20,100,200])['per_channel']['bm25']['recall']
r_enr=recall_ceiling({'bm25':enrl},golds, ks=[20,100,200])['per_channel']['bm25']['recall']
print('BM25 recall@100  raw=',round(r_raw[100],3),' enriched=',round(r_enr[100],3),
      ' (gate: enriched should lift recall; else drop A1)')

## 7. FAST decision: gold-targeted recall A/B (no full 47k run needed)
Enrich ONLY the dev-gold tracks (~hundreds) and measure raw-vs-enriched BM25 recall on those dev sessions. Decisive + cheap. (Slightly optimistic since only golds are enriched, but a clear directional signal: does enrichment make the gold findable by its real query? If yes -> do the full run via Batch/direct key.)

In [ ]:
from mcrs.data.conversations import Conversations
from mcrs.data.catalog import Catalog
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.eval.probe import recall_ceiling
from concurrent.futures import ThreadPoolExecutor

AB_SESSIONS = 200
dev = load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='test').select(range(AB_SESSIONS))
conv = Conversations(dev)
turns = list(conv.turns())
golds = [conv.gold(t.session_id, t.turn_number) for t in turns]
by_id = {canonical_track_id(r['track_id']): r for r in rows}
gold_ids = sorted({g for g in golds if g and g in by_id})
print(len(gold_ids), 'unique gold tracks to enrich (this is the only Gemini cost here)')

def _enr(tid):
    try: return tid, enriched_document(by_id[tid], gen(by_id[tid]))
    except Exception: return tid, None
gold_enr = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    for tid, doc in ex.map(_enr, gold_ids):
        if doc: gold_enr[tid] = doc
print('enriched', len(gold_enr), 'gold tracks')

qs = [QueryBuilder().build(t).text for t in turns]
raw = BM25Channel(Catalog(rows)).batch_text_to_item_retrieval(qs, 200)
enr = BM25Channel(Catalog(rows, enriched_docs=gold_enr), enriched=True).batch_text_to_item_retrieval(qs, 200)
rr = recall_ceiling({'bm25': raw}, golds, ks=[20,100,200])['per_channel']['bm25']['recall']
re_ = recall_ceiling({'bm25': enr}, golds, ks=[20,100,200])['per_channel']['bm25']['recall']
print('\nBM25 recall (gold-targeted A/B):')
for k in (20,100,200): print(f'  @{k}: raw={rr[k]:.3f}  enriched={re_[k]:.3f}  delta={re_[k]-rr[k]:+.3f}')
print('\nGATE: enriched should clearly beat raw -> A1 worth the full run; else tune/drop.')